In [8]:
%env CUDA_VISIBLE_DEVICES=0
import json
import os

import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm
import yadisk
import dotenv

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel, AutoTokenizer, XCLIPTextModel

env: CUDA_VISIBLE_DEVICES=0


### Получение эмбеддингов

In [9]:
# https://huggingface.co/docs/transformers/main/model_doc/xclip

video_model = AutoModel.from_pretrained("microsoft/xclip-base-patch32")
processor = AutoProcessor.from_pretrained("microsoft/xclip-base-patch32")

text_model = XCLIPTextModel.from_pretrained("microsoft/xclip-base-patch32")
tokenizer = AutoTokenizer.from_pretrained("microsoft/xclip-base-patch32")

DEVICE = 'cuda:0'
text_model.to(DEVICE)
video_model.to(DEVICE)

XCLIPModel(
  (text_model): XCLIPTextTransformer(
    (embeddings): XCLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): XCLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x XCLIPEncoderLayer(
          (self_attn): XCLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): XCLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps

In [10]:
def encode_frame(frame):
    return torch.tensor([0.1,0.2,0.3])

def encode_video(frames):
    # внутри 1280 x 720 -> 224 x 224
    with torch.no_grad():
        while len(frames) < 8:
            frames.append(frames[-1])
        inputs = processor(videos=[frames], return_tensors="pt").to(DEVICE)
        video_features = video_model.get_video_features(**inputs)
        video_features = video_features.to('cpu')
        inputs = inputs.to('cpu')
        torch.cuda.empty_cache()
    return video_features[0]

def encode_text(text):
    with torch.no_grad():
        inputs = tokenizer([text], 
                        truncation=True,
                        padding=True, 
                        return_tensors="pt").to(DEVICE)
        outputs = text_model(**inputs).pooler_output.to('cpu')
        del inputs
        return outputs

Делать для каждого видео свой кадровый шаг исходя из того, какой там изначально fps. У нас новый fps должен быть 3 кадра / секунду

In [11]:
def find_sub_vid_overlap(sub_s, sub_e, vid_s, vid_e, min_overlap=3):
    sub = [i for i in range(int(sub_s), int(sub_e)+1, 1)]
    vid = [i for i in range(int(vid_s), int(vid_e)+1, 1)]
    overlap = len(set(sub) & set(vid))
    if overlap == len(sub):
        return True
    elif overlap > min_overlap:
        return True
    return False

In [ ]:
def extract_frames(cap, sub_df, reuse_frames=True,
                   window_size=11, overlap=5, new_fps=3):
    '''
    sub_df = dataframe with columns "text", "start", "end"
    overlap = non-overlap between video pieces in seconds (sliding window step)
    window_size = size of video pieces in seconds
    new_fps = fps we read the video with

    if no overlap needed, just set overlap = window_size
    '''

    video_embeddings = [] # эмбеддинг видео кусочка
    text_embeddings = [] # эмбеддинг соответствующего текста
    seconds = []
    texts = []

    # определим кадры, которые пойдут в наши кусочки
    good_frames_ids = []
    fps = cap.get(cv2.CAP_PROP_FPS) # собственное fps видео
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_step = fps // new_fps # например, 25//3 = 8

    for i in range(0, int(frame_count), int(fps)):
        good_frames_ids.extend([i+k*frame_step for k in range(new_fps)])

    good_frames_ids = set(good_frames_ids) # для ускорения поиска
    # прочитаем эти кусочки
    good_frames = [] # это возможно не считается на маломощной машине
    frame_embeddings = []
    m = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if m in good_frames_ids:
            # .to_ndarray(format="rgb24")
            if reuse_frames:
                frame_embeddings.append(encode_frame(frame))
            else:
                good_frames.append(frame)
        m += 1

    # теперь идём по ним с нужным шагом и вынимаем субтитры
    for step in range(0, max(len(good_frames), len(frame_embeddings)), new_fps*overlap):
        seconds_start = step / new_fps
        seconds_end = seconds_start + window_size

        frame_sub_df = sub_df[(((sub_df['start'] > seconds_start) & # начало текста внутри видео
                                (sub_df['start'] < seconds_end)) |
                                ((sub_df['end'] > seconds_start) & # конец текста внутри видео
                                (sub_df['end'] < seconds_end)) |
                                ((sub_df['end'] > seconds_end) & # видео внутри текста
                                (sub_df['start'] < seconds_start)) 
                                )]
        subs = ''
        for _, row in frame_sub_df.iterrows():
            if find_sub_vid_overlap(row['start'], row['end'], 
                                        seconds_start, seconds_end, 
                                        min_overlap=2):
                subs += row['text'] + ' '
            
        frames_start = step
        frames_end = frames_start + window_size*new_fps
        if reuse_frames:
            initial_frames_emb = frame_embeddings[frames_start:frames_end]
            frames_emb = torch.concatenate(initial_frames_emb)
            frames_emb = frames_emb.reshape(len(initial_frames_emb), len(initial_frames_emb[0]))
            frames_emb = torch.mean(frames_emb, axis=0)
        else:
            frames_emb = encode_video(good_frames[frames_start:frames_end])

        text_embeddings.append(encode_text(subs.strip(' ')))
        video_embeddings.append(frames_emb)
        seconds.append([seconds_start, seconds_end])
        texts.append(subs)
        torch.cuda.empty_cache()
        del frames_emb

    return seconds, video_embeddings, text_embeddings, texts

In [13]:
#  было бы неплохо привести это в норм вид потом для публичного датасета 
def read_subtitles(sub_folder_path, sub_mapping_file_path):
    id2name = json.load(open(sub_mapping_file_path, 'r', encoding='utf-8'))
    bad_symbols = '?\":/|'

    caption_dict = dict() # Название видео : DataFrame(columns=["text", "start", "end"])
    #new_caption_dict = dict()
    for caption_file in tqdm([sf for sf in os.listdir(sub_folder_path)]):
        fn = sub_folder_path+ caption_file
        this_cap_list = []
        with open(fn, 'r', encoding='utf-8') as f:
            for caption in json.load(f):
                cap_dict = caption.copy()
                this_cap_list.append({'text': cap_dict['text'], 
                                    'start': cap_dict['start'],
                                    'end': cap_dict['start'] + cap_dict['duration']})
        
        name = caption_file.split('/')[-1].split('.')[0]
        video_name = id2name[name]+'.mp4'
        for bs in bad_symbols:
            video_name = video_name.replace(bs, '')
        if video_name == 'Интервью с двукратным сурдлимпийским чемпионом Владиславом Винником. 2 часть.mp4':
            video_name = "Интервью с двукратным сурдлимпийским чемпионом Владиславом Винником. 2 часть. С субтитрами.mp4"
        caption_dict[video_name] = pd.DataFrame(data=this_cap_list)
    return caption_dict

ГЛАВНАЯ ЯЧЕЙКА!!!

In [14]:
# Все видео (316 по итогу)
no_subs_set = set() # видео без субтитров нас не интересуют
with open('../dataset/no_subs.json', 'r', encoding='utf-8') as f:
    no_subs = json.load(f)
    for no_sub in no_subs:
        no_subs_set.add(no_sub['vid_path'].replace('\"', ''))

sub_folder_path = '../dataset/subs/'
sub_mapping_file_path = '../dataset/id2name.json'
caption_dict = read_subtitles(sub_folder_path, sub_mapping_file_path)
print(len(caption_dict))


video_folder_path = '../dataset/videos/'
download = False

files = os.listdir(video_folder_path) # или list(client.listdir("disk:/SLR Project"))
for file in tqdm(files[:2]):
    if download:
        video_name = file['path'].split('/')[-1]
        path_on_disk = file['path']
        path_to_store = './' + video_name
        # и прописать сюда скачивание
        pass
    else:
        video_name = file.split('/')[-1]
    if video_name not in no_subs_set and video_name in caption_dict:
        sub_df = caption_dict[video_name]
        
        cap = cv2.VideoCapture(video_folder_path + file)
        seconds, video_embeddings, text_embeddings, texts = extract_frames(cap, sub_df,
                                                                           window_size=4,
                                                                           new_fps=2,
                                                                           overlap=4,
                                                                           reuse_frames=False)
        torch.save([video_embeddings, text_embeddings], f'../dataset/processed/embeddings/{video_name}.pt')
        with open(f'../dataset/processed/info/{video_name}.json', 'w', encoding='utf-8') as f:
            json.dump([seconds, texts], f, ensure_ascii=False, indent=4)
        
        # а разбиение на трейн и тест?))))) сделаем при загрузке для обучения может быть...

    if download:
        # удалить скачанное видео
        pass

  0%|          | 0/321 [00:00<?, ?it/s]

100%|██████████| 321/321 [00:00<00:00, 1665.01it/s]


321


100%|██████████| 2/2 [01:15<00:00, 37.81s/it]


In [ ]:
# проверяем, что всё сохранилось
vembs, tembs = torch.load('/home/user/alina_science/rsl/dataset/processed/embeddings/Фестиваль Искусств Палитра тишины. С субтитрами.mp4.pt')
print(len(tembs))
print(len(vembs))

with open('/home/user/alina_science/rsl/dataset/processed/info/Какие вопросы будут обсуждаться на семинарах.mp4.json',
          'r') as f:
    seconds, subs = json.load(f)
print(len(seconds))
print(len(subs))

218


### Test / train split и dataloader

- последние 10%
- случайные 10% до этого
- и исключить все кусочки с пустыми субтитрами

10% из 90% - это каждый девятый кусочек

0 1 2 3 4 5 6 7 |8| 9 10 11 12 13 14 15 16 |17| | 18 19

In [1]:
from datasets import Dataset
import os 
import torch
import json
from tqdm import tqdm
import numpy as np

/home/user/alina_science/rsl_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class RSL_Dataset(): # Dataset
    def __init__(self):
        self.video_embs = []
        self.text_embs = []
        self.texts = []
        
    def __len__(self):
        return len(self.video_embs)

    def __getitem__(self, idx):
        return self.video_embs[idx], self.text_embs[idx], self.texts[idx]
    
    def add_item(self, video_emb, text_emb, text):
        self.video_embs.append(video_emb.squeeze())
        self.text_embs.append(text_emb.squeeze())
        self.texts.append(text)

In [3]:
# тестовые видео оставляем теми же
test_videos = set()
with open('../dataset/test.json', 'r') as f:
    tvds = json.load(f)
    for tvd in tvds:
        test_videos.add(tvd['video'])

In [4]:
emb_dir = '/home/user/alina_science/rsl/dataset/processed/embeddings/'
info_dir = '/home/user/alina_science/rsl/dataset/processed/info/'

train_dataset = RSL_Dataset()
eval_dataset = RSL_Dataset()
test_dataset = RSL_Dataset()

for emb_file in tqdm(os.listdir(emb_dir)):
    if emb_file[:-3] in test_videos:
        with open(info_dir + '/' + emb_file[:-3]+'.json', 'r') as f:
            info = json.load(f)
        texts = info[1]
        for t, temb, vemb in zip(texts, tembs, vembs):
            if t != '':
                test_dataset.add_item(vemb, temb, t)

    else:
        with open(info_dir + '/' + emb_file[:-3]+'.json', 'r') as f:
            info = json.load(f)
        texts = info[1]
        vembs, tembs = torch.load(emb_dir + '/' + emb_file)

        good_vembs = []
        good_tembs = []
        good_texts = []
        for t, temb, vemb in zip(texts, tembs, vembs):
            if t != '':
                good_vembs.append(vemb)
                good_tembs.append(temb)
                good_texts.append(t)

        ten_last_i = int(len(good_tembs)*0.9)
        for i, e in enumerate(zip(good_tembs, good_vembs, good_texts)):
            if i > 0 and (i%9 == 0 or i > ten_last_i):
                eval_dataset.add_item(e[0], e[1], e[2])
            else:
                train_dataset.add_item(e[0], e[1], e[2])

100%|██████████| 316/316 [00:10<00:00, 29.28it/s]


In [5]:
print(len(train_dataset), len(eval_dataset), len(test_dataset))

28380 6754 2041


### Пробуем подключить обучение

In [6]:
from torch import nn
from torch.nn.functional import log_softmax
import torch
from torch.utils.data import DataLoader, Dataset
from torchmetrics.functional import pairwise_cosine_similarity
from open_clip import ClipLoss # встроить в код!!!
import wandb

In [7]:
wandb.login(key='3c8ca549cb679c1d77de67d8b9d1e5684351f8e8')

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/user/.netrc
wandb: Currently logged in as: alinarechina to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
class MLPClassifier(nn.Module):
    def __init__(
          self,
          vids_emb_dim,
          text_emb_dim,
          hidden_layer_dim=2048,
          dropout_p=0.2,
          n_layers=5,
        ):
        super().__init__()

        self.vids_head = nn.ModuleList()
        self.vids_head.append(nn.Linear(vids_emb_dim, hidden_layer_dim))
        for i in range(n_layers - 1):
            self.vids_head.append(nn.ReLU())
            self.vids_head.append(nn.Dropout(dropout_p))
            self.vids_head.append(nn.Linear(hidden_layer_dim, hidden_layer_dim))
        self.vids_head = nn.Sequential(*self.vids_head)
        

        self.text_head = nn.ModuleList()
        self.text_head.append(nn.Linear(text_emb_dim, hidden_layer_dim))
        for i in range(n_layers - 1):
            self.text_head.append(nn.ReLU())
            self.text_head.append(nn.Dropout(dropout_p))
            self.text_head.append(nn.Linear(hidden_layer_dim, hidden_layer_dim))
        self.text_head = nn.Sequential(*self.text_head)
    
    def forward(self, vids_emb, text_emb):
        return (self.vids_head(vids_emb), 
                self.text_head(text_emb))

In [9]:
epoch_num = 10
lr = 0.01
hidden_layer_dim = 2048
dropout_p = 0.2
batch_size = 32
n_layers = 2

In [8]:
def contrastive_loss(logits, dim):
    neg_ce = torch.diag(log_softmax(logits, dim=dim))
    return -neg_ce

def clip_loss(similarity: torch.Tensor) -> torch.Tensor: # заменить на ClipLoss!!
    caption_loss = contrastive_loss(similarity, dim=0)
    image_loss = contrastive_loss(similarity, dim=1)
    return (caption_loss + image_loss) / 2.0

def metrics(similarity: torch.Tensor):
    y = torch.arange(len(similarity)).to(similarity.device)
    cap2img_match_idx = similarity.argmax(dim=0)
    cap_acc = (cap2img_match_idx == y).float()

    return cap_acc

In [11]:
emb_dim = 512
model = MLPClassifier(
    vids_emb_dim=emb_dim,
    text_emb_dim=emb_dim,
    hidden_layer_dim=hidden_layer_dim,
    dropout_p=dropout_p,
    n_layers=n_layers,
)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [12]:
logging = wandb.init(
    project="RSL",
    name="test",
    config={
        "learning_rate": lr,
        "epochs": epoch_num,
        "hidden_layers_dim": hidden_layer_dim,
        "dropout_rate": dropout_p,
        "batch_size": batch_size,
        "n_layers": n_layers
    }
)

In [13]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
eval_dataloader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
for epoch in range(epoch_num):
    for vids_emb, text_emb, text in tqdm(train_dataloader):
        optimizer.zero_grad()
        #pred = model(vids_emb, text_emb)
        #softmax(single_img_emb @ text_emb_db.T, dim=-1)
        encoded_vids, encoded_texts = model(vids_emb, text_emb)
        pred = pairwise_cosine_similarity(encoded_vids, encoded_texts) 
        loss = clip_loss(pred).mean()
        logging.log({"train_loss": loss.item()})
        loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1e5)
        optimizer.step()

    with torch.no_grad():
        model.eval()
        loss = torch.tensor([])
        accuracy = torch.tensor([])
        for vids_emb, text_emb, text in tqdm(eval_dataloader):
            encoded_vids, encoded_texts = model(vids_emb, text_emb)
            pred = pairwise_cosine_similarity(encoded_vids, encoded_texts)
            pred_norm = (pred + 1) / 2
            target = torch.diag(torch.ones(vids_emb.shape[0]))
            loss = torch.concat((loss, clip_loss(pred)), dim=0)
            accuracy = torch.concat((accuracy, metrics(pred)), dim=0)

        logging.log({
            "val_loss": loss.mean().item(),
            "val_acc": accuracy.mean().item()
        })

        torch.save(model.state_dict(), f'./checkpoints/mlp_2layers_{epoch}.pt')
        model.train()
        print('loss', loss.mean().item())
        print('val_acc', accuracy.mean().item())

100%|██████████| 212/212 [00:01<00:00, 169.86it/s]


loss 3.248901605606079
val_acc 0.10082913935184479


100%|██████████| 212/212 [00:01<00:00, 169.45it/s]


loss 3.194805860519409
val_acc 0.1017175018787384


100%|██████████| 212/212 [00:01<00:00, 165.34it/s]


...

loss 3.1476845741271973
val_acc 0.14435890316963196


100%|██████████| 212/212 [00:01<00:00, 166.10it/s]


loss 3.134061813354492
val_acc 0.1495410054922104


100%|██████████| 212/212 [00:01<00:00, 156.70it/s]


loss 3.1405231952667236
val_acc 0.14569143950939178


In [ ]:
# тест
model.eval()
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) 
loss = torch.tensor([])
accuracy = torch.tensor([])

diag = 0
full_sum = 0
with torch.no_grad():
    for vids_emb, text_emb, text in tqdm(test_dataloader): # тут два батча
            encoded_vids, encoded_texts = model(vids_emb, text_emb)
            pred = pairwise_cosine_similarity(encoded_vids, encoded_texts)
            pred_norm = (pred + 1) / 2
            target = torch.diag(torch.ones(vids_emb.shape[0]))
            loss = torch.concat((loss, clip_loss(pred)), dim=0)
            accuracy = torch.concat((accuracy, metrics(pred)), dim=0)

            pred_norm = pred_norm.detach().numpy()

            diag += np.trace(pred_norm)
            full_sum += np.sum(pred_norm)

    out = full_sum.item() - diag

    print('loss', loss.mean().item())
    print('test_acc', accuracy.mean().item())
    print('diag_sum', diag)
    print('out_sum', out)

100%|██████████| 64/64 [00:00<00:00, 143.19it/s]


loss 3.507270574569702
test_acc 0.03037726692855358
diag_sum 1134.0237
out_sum 34809.695


- Посмотреть глазами на инференс, мб оно случайное
- Сумма на диагонали не может быть больше 32, иначе не нормализовано!!
- Усреднить диагональ, усреднить не диагональ

### Для уже обученных моделек

In [20]:
# измеряем качество лушей модели на тесте
import numpy as np

text2pred = dict()

emb_dim = 512
hidden_layer_dim = 4096
dropout_p = 0.2
batch_size=32
model_0 = MLPClassifier(
    vids_emb_dim=emb_dim,
    text_emb_dim=emb_dim,
    hidden_layer_dim=hidden_layer_dim,
    dropout_p=dropout_p,
    n_layers=5,
)
# model_0.load_state_dict(torch.load('/home/user/alina_science/rsl/src/checkpoints/mlp_5layers_24.pt', 
#                                    weights_only=True))
loaded_dict = torch.load('/home/user/alina_science/rsl/src/checkpoints/mlp_5layers_bs32_lr0@001_drp0@2_dim4096/29.pt')
adapted_dict = {}
for p in loaded_dict.keys():
    adapted_dict[p.replace("emb_layer", "head")] = loaded_dict[p]
model_0.load_state_dict(adapted_dict)

test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) 
model_0.eval()
loss = torch.tensor([])
accuracy = torch.tensor([])

diags = []
outs = []
with torch.no_grad():
    for vids_emb, text_emb, text in tqdm(test_dataloader): # тут два батча
            encoded_vids, encoded_texts = model_0(vids_emb, text_emb)
            pred = pairwise_cosine_similarity(encoded_vids, encoded_texts)
            target = torch.diag(torch.ones(vids_emb.shape[0]))
            loss = torch.concat((loss, clip_loss(pred)), dim=0)
            accuracy = torch.concat((accuracy, metrics(pred)), dim=0)

            pred_norm = (pred + 1) / 2
            pred_norm = pred_norm.detach()
            diags.append(pred_norm.diag().sum()/pred_norm.size(0))
            outs.append(pred_norm[~torch.eye(pred_norm.size(0), dtype=torch.bool, 
                                             device=pred_norm.device)].sum()/
                                             (pred_norm.size(0)-1)/pred_norm.size(1))

            # пытаемся смотреть что внутри
            topk_values, topk_indices = torch.topk(pred, k=2, dim=1)
            for vid_idx in range(pred.shape[0]):
                text2pred[text[vid_idx]] = [text[ti] for ti in topk_indices[vid_idx]]


    print('loss', loss.mean().item())
    print('test_acc', accuracy.mean().item())
    print('diag_sum', sum(diags)/len(diags))
    # print('diag_sum_ratio', sum(diags)/len(diags)/32)
    print('out_sum', sum(outs)/len(outs))
    # print('out_sum_ratio', sum(outs)/len(outs)/(32*31))

100%|██████████| 64/64 [00:04<00:00, 15.92it/s]

loss 3.5420420169830322
test_acc 0.03086722269654274
diag_sum tensor(0.4499)
out_sum tensor(0.4538)


In [21]:
list(text2pred.items())[:3]

[('Иван Иванович Снетков, жестовое имя "свитер", ',
  ['Снетков содействовал развитию работы в организациях ВОГ \nв Иркутске, Красноярске, Томске, Новосибирске.  ',
   'и одновременно заведующим оргсектором\n Западносибирского краевого отдела социального обеспечения. ']),
 ('Иван Иванович Снетков, жестовое имя "свитер", потому что он всегда одевался представительно - пиджак и свитер. ',
  ['где учился в 1929–30 гг. и был секретарем ячейки ВЛКСМ. ',
   'В 1929 году работал в московской промартели «Везувий» штамповщиком.  ']),
 ('потому что он всегда одевался представительно - пиджак и свитер. ',
  ['В 1929 году работал в московской промартели «Везувий» штамповщиком.  ',
   'В июле 1927 году в 18-летнем возрасте заболел менингитом и потерял слух (чистую речь он, естественно, сохранил).  '])]

In [22]:
list(text2pred.items())[100:103]

[('Мать Ромы, слышащая, позвонила в комнату 39. Чтобы узнать, говорю ли я правду. ',
  ['И мы встретились, пошли. Нажали на кнопки домофона и начали подниматься на 4-й этаж по ступенькам. ',
   'Вышел старенький мужчина и обрадовался, увидев нас, ']),
 ('Вышел старенький мужчина и обрадовался, увидев нас, ',
  ['И мы встретились, пошли. Нажали на кнопки домофона и начали подниматься на 4-й этаж по ступенькам. ',
   'Вышел старенький мужчина и обрадовался, увидев нас, ']),
 ('он знал меня и Тоню: «Давно не виделись. Ах!». ',
  ['И теперь он один в большой комнате.\nМама Ромы успокоилась: не вру я и хищных намерений не имею. ',
   'И цветы кругом – все как раньше! Сильно удивилась: ничего не изменилось. '])]

- средняя сумма логитов на диагонали (должна расти)
- средняя сумма логитов вне диагонали (должна падать)

- можно использовать softmax
- аккумуляция градиентов
- увеличить размер батча